# ex02 · 从零实现训练循环（对应教材 3.2 线性回归的从零开始实现）

> **做题流程**：按「第一部分 → 第五部分」顺序，补全每个 TODO 后运行对应的自测 cell。
> 测试没跑过时只会提示「⚠ 先完成 TODO x」，不会报红错；做完再看 `solutions/ex02-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节是**全书第一个完整训练循环**：数据迭代器 → 模型 → 损失 → 优化器 四件套。
> 考点：zero_grad → backward → step 的顺序。

In [1]:
import random
import torch

## 第一部分 · 生成数据集（给定代码，运行即可）

用带噪声的线性模型造 1000 个样本：真实参数 w = [2, −3.4]、b = 4.2，噪声标准差 0.01。
运行后回答：features 的形状是 ____，labels 的形状是 ____，为什么 labels 要把他多出一个维度？

为了行行对应 + 方便后续矩阵运算对齐形状（避免错误广播）

**【你的预测】**

In [2]:
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)
print('features 形状:', list(features.shape))
print('labels 形状:', list(labels.shape))
print('第 0 个样本:', features[0].tolist(), ' 标签:', labels[0, 0].item())

features 形状: [1000, 2]
labels 形状: [1000, 1]
第 0 个样本: [-0.8942933082580566, 1.5385587215423584]  标签: -2.8324198722839355


## 第二部分 · 数据迭代器（TODO 2.1 ~ 2.2）

按小批量把数据喂给模型。骨架已搭好，补全两处 TODO：

- 每个 epoch 都要随机打乱样本顺序
- 按 batch_size 切片，把每批的 features / labels 用 yield 交出去（提示：切片索引需要转成 torch.tensor）

In [3]:
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    # TODO 2.1: 生成 0..num_examples-1 的索引列表，并随机打乱
    indices = list(range(num_examples))
    # 这些样本是随机读取的，没有特定的顺序
    random.shuffle(indices)
    # TODO 2.2: 从 0 开始按 batch_size 步进，切出每批索引（注意最后一批可能不满），
    #           用 yield 返回 (features[batch_indices], labels[batch_indices])
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

### 自测：完成 TODO 2.1~2.2 后运行

In [4]:
total = 0
shapes_ok = True
try:
    for X, y in data_iter(10, features, labels):
        shapes_ok = shapes_ok and (list(X.shape) == [10, 2]) and (list(y.shape) == [10, 1])
        total += len(X)
    assert shapes_ok and total == 1000, f'shapes_ok={shapes_ok}, total={total}'
    print('✓ data_iter：每批 (10, 2)，总共 1000 个样本')
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ data_iter：每批 (10, 2)，总共 1000 个样本


## 第三部分 · 模型、损失、优化器（TODO 2.3 ~ 2.5）

三个函数各补一处 TODO：模型（矩阵乘法+偏置）、均方损失、小批量随机梯度下降。
注意 sgd 里要做两件事：更新参数、清零梯度——顺序不能反。

In [5]:
def linreg(X, w, b):
    # TODO 2.3: 返回 X @ w + b
    return X @ w + b
    

def squared_loss(y_hat, y):
    # TODO 2.4: 返回 (y_hat - y)^2 / 2
    # 提示：y 的形状是 (batch_size, 1)，先 reshape 成与 y_hat 相同再相减
    return (y_hat - y.reshape(-1, 1)) ** 2 / 2


def sgd(params, lr, batch_size):
    # TODO 2.5: 在 torch.no_grad() 里更新每个参数：param -= lr * param.grad / batch_size，
    #           更新之后立刻 param.grad.zero_()
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()


### 自测：完成 TODO 2.3~2.5 后运行

In [6]:
try:
    Xt = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
    wt = torch.tensor([[0.5], [-1.0]])
    bt = torch.tensor([2.0])
    pred = linreg(Xt, wt, bt)
    assert list(pred.shape) == [2, 1], f'预测形状不对: {pred.shape}'
    print('✓ linreg 输出:', pred.flatten().tolist())

    yt = torch.tensor([[0.0], [-1.0]])
    ls = squared_loss(pred, yt)
    print('✓ squared_loss 输出:', ls.flatten().tolist())

    p = torch.tensor([2.0], requires_grad=True)
    (p * 3.0).backward()
    sgd([p], lr=0.1, batch_size=1)
    assert abs(p.item() - 1.7) < 1e-6, f'sgd 更新不对: {p.item()}'
    assert p.grad.item() == 0.0, 'sgd 没有清零梯度'
    print('✓ sgd 更新与清零正确')
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ linreg 输出: [0.5, -0.5]
✓ squared_loss 输出: [0.125, 0.125]
✓ sgd 更新与清零正确


## 第四部分 · 训练循环（TODO 2.6 ~ 2.8）——全书第一个训练循环

四件套就位后，把它们串起来。补全训练循环的三个 TODO。
先思考（考点，写你的答案）：

- 为什么是 l.sum().backward() 而不是 l.backward()？
- 梯度清零发生在哪个函数里？如果把 sgd 里的 zero_() 删掉会怎样？
- 清零 → 反向 → 更新 这三步的顺序能调换吗？

In [8]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.03
num_epochs = 3
batch_size = 10

try:
    for epoch in range(num_epochs):
        for X, y in data_iter(batch_size, features, labels):
            # TODO 2.6: 计算这个小批量的损失（用 linreg 和 squared_loss）
            y_pred = linreg(X, w, b)
            loss = squared_loss(y, y_pred)
            # TODO 2.7: 反向传播（损失形状是 (batch_size, 1)，先求和变成标量）
            loss.sum().backward()
            # TODO 2.8: 用 sgd 更新 [w, b]
            sgd([w, b], lr, batch_size)
        with torch.no_grad():
            train_l = squared_loss(linreg(features, w, b), labels)
            print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')
except NotImplementedError as e:
    print(f'⚠ {e}，先完成 TODO 2.6~2.8 再运行')

epoch 1, loss 0.037945
epoch 2, loss 0.000153
epoch 3, loss 0.000049


## 第五部分 · 训练结果检验

先预测再运行：

- 3 个 epoch 后 w 和 b 会接近真值 [2, −3.4] 和 4.2 吗？误差大约多大？
- 3 个 epoch 的 loss 是递增还是递减？

**【你的预测】**

In [9]:
print('估计的 w:', w.reshape(1, -1).detach().numpy().round(4))
print('估计的 b:', round(b.item(), 4))
print('真实 w: [2.0, -3.4]   真实 b: 4.2')
print('w 的逐项误差:', (w.detach().reshape(-1) - true_w).abs().tolist())

估计的 w: [[ 2.0006 -3.3993]]
估计的 b: 4.199
真实 w: [2.0, -3.4]   真实 b: 4.2
w 的逐项误差: [0.0005903244018554688, 0.0006527900695800781]


## 小结与面试衔接

- 训练循环四件套：数据迭代器 → 模型 → 损失 → 优化器
- 三步顺序：清零梯度 → 反向传播 → 更新参数（清零发生在 sgd 内部）
- l.sum().backward()：把 batch 的损失向量求和成标量再反向
- sgd 里除以 batch_size：让学习率的尺度不随批量大小变化
- 面试高频：梯度不清零会怎样（累加）、为什么不能全零初始化（ch04 ex05 展开）